# 04 - Time Series Analysis and Analytical Report

Build a trend view, create a simple forecast, and write report-ready outputs.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.linear_model import LinearRegression

from analysis_utils import ensure_output_dir, get_pyodbc_connection, load_indicator_data

ENV_FILE = ".env"  # Windows learners can change this to ".env.windows"
sns.set_theme(style="whitegrid")
output_dir = ensure_output_dir()
df = load_indicator_data(ENV_FILE)
df.head()

## Daily Trend and Rolling Average

In [ ]:
daily = (
    df.groupby("ObservationDate", as_index=False)
    .agg(
        LiquidityRatio=("LiquidityRatio", "mean"),
        NplRatio=("NplRatio", "mean"),
        TransactionValueLSL=("TransactionValueLSL", "sum"),
        StressRate=("StressFlag", "mean"),
    )
    .sort_values("ObservationDate")
)

daily["LiquidityRolling7"] = daily["LiquidityRatio"].rolling(7, min_periods=1).mean()
daily["DayIndex"] = np.arange(len(daily))
daily.tail()

## Simple Trend Forecast

In [ ]:
trend_model = LinearRegression()
trend_model.fit(daily[["DayIndex"]], daily["LiquidityRatio"])

future_days = pd.DataFrame({"DayIndex": np.arange(len(daily), len(daily) + 7)})
last_date = daily["ObservationDate"].max()
future_days["ObservationDate"] = pd.date_range(last_date + pd.Timedelta(days=1), periods=7)
future_days["ForecastLiquidityRatio"] = trend_model.predict(future_days[["DayIndex"]])

future_days.to_csv(output_dir / "liquidity_forecast.csv", index=False)
future_days

## Forecast Chart

In [ ]:
fig, ax = plt.subplots(figsize=(11, 5))
sns.lineplot(data=daily, x="ObservationDate", y="LiquidityRatio", ax=ax, label="Observed", color="#0F766E")
sns.lineplot(data=daily, x="ObservationDate", y="LiquidityRolling7", ax=ax, label="7-day rolling average", color="#7C3AED")
sns.lineplot(data=future_days, x="ObservationDate", y="ForecastLiquidityRatio", ax=ax, label="7-day forecast", color="#DC2626", linestyle="--")
ax.set_title("Liquidity Ratio Trend and Forecast")
ax.set_xlabel("Observation date")
ax.set_ylabel("Liquidity ratio")
fig.autofmt_xdate()
fig.tight_layout()
fig.savefig(output_dir / "liquidity_forecast.png", dpi=160)
plt.show()

## Report Summary and SQL Log

In [ ]:
metrics_path = output_dir / "model_metrics.csv"
model_accuracy = None
if metrics_path.exists():
    model_accuracy = float(pd.read_csv(metrics_path).loc[0, "accuracy"])

observation_rows = int(len(df))
mean_liquidity = float(df["LiquidityRatio"].mean())
mean_npl = float(df["NplRatio"].mean())
stress_rate = float(df["StressFlag"].mean())

report_text = f"""# Module 5 Analytical Report Summary

Observation rows: {observation_rows}

Mean liquidity ratio: {mean_liquidity:.4f}

Mean NPL ratio: {mean_npl:.4f}

Stress observation rate: {stress_rate:.2%}

Model accuracy: {model_accuracy if model_accuracy is not None else 'Run notebook 03 first'}

Generated visuals:

- outputs/liquidity_trend.png
- outputs/npl_distribution_by_type.png
- outputs/correlation_heatmap.png
- outputs/stress_classifier_confusion_matrix.png
- outputs/liquidity_forecast.png
"""

Path(output_dir / "module5_report_summary.md").write_text(report_text, encoding="utf-8")

with get_pyodbc_connection(ENV_FILE) as conn:
    cursor = conn.cursor()
    cursor.execute(
        """
        INSERT INTO m5.AnalysisReportRun
            (AnalystName, ObservationRows, MeanLiquidityRatio, MeanNplRatio, ModelAccuracy, Notes)
        VALUES (?, ?, ?, ?, ?, ?);
        """,
        "Learner",
        observation_rows,
        mean_liquidity,
        mean_npl,
        model_accuracy,
        "Notebook-generated Module 5 analytical report summary",
    )
    conn.commit()

print(report_text)